In [1]:
from fasthtml.common import *
from fasthtml.jupyter import *

In [2]:
# hx_live_hdr = Script(src='https://cdn.jsdelivr.net/npm/htmx.org@next/dist/ext/hx-live.min.js')

In [3]:
app, rt = fast_app(htmx=False, htmx4=True, exts='live')

# Trigger

In [4]:
@rt("/")
def get():
    i = Input(id="name", value="world")
    o = Output(hx_live="this.textContent='hello, ' + q('#name').value")
    return Div(i, o, hx_ext="hx-live")

srv = JupyUvi(app)

In [8]:
srv.stop()

In [9]:
@rt("/")
def get():
    i = CheckboxX(id="tick", value="tick")
    o = Output(
        hx_live="""
            let term = q('#tick').checked;
            if (term) {
                this.textContent = "Hello";
            } else {
                this.textContent= '';
            }

            return;
        """
    )
    return Div(i), Div(o)

In [10]:
srv = JupyUvi(app)

In [11]:
srv.stop()

# Swap Coordination

In [12]:
@rt("/")
def get():
    total = Output(
        "0",
        id="total",
        hx_live="""
            let nums = q('.num').arr().map(x => Number(x.value || 0));
            this.textContent = nums.reduce((a, b) => a + b, 0);
        """
    )
    return Div(
        Button("Load numbers", hx_get="/numbers", hx_target="#numbers", hx_swap="innerHTML"),
        Div(id="numbers"),
        P("Total: ", total),
    )

@rt("/numbers")
def get():
    return Div(
        Input(type="number", cls="num", value="10"),
        Input(type="number", cls="num", value="20"),
        Input(type="number", cls="num", value="30"),
    )

srv = JupyUvi(app)

In [13]:
srv.stop()

# Scope Helpers

Timeout: it's not cancel so lots of triggering will create a long queues

In [34]:
@rt("/")
def get():
    i = Input(id="name", value="world")
    o = Output(
        hx_live="""
            await timeout(3000);
            this.textContent='hello, ' + q('#name').value;
        """
        )
    return Div(i, o, hx_ext="hx-live")


In [ ]:

srv = JupyUvi(app)

In [35]:
srv.stop()

Bounce

In [36]:
@rt("/")
def get():
    i = Input(id="name", value="world")
    o = Output(
        hx_live="""
            await debounce(3000);
            this.textContent='hello, ' + q('#name').value;
        """
        )
    return Div(i, o, hx_ext="hx-live")


In [37]:

srv = JupyUvi(app)

In [14]:
srv.stop()

# Reproduce htmx examples using fasthtml

From `15-hx-live.md`

### Derived value

In [ ]:
@rt('/')
def get():
    price = Label("Price: ", Input(id="price", value="10", type="number"))
    quantity = Label("Quantity: ", Input(id="qty", value="3", type="number"))
    o = Label(
        "Total:",
        P(hx_live="this.textContent = q('#price').valueAsNumber * q('#qty').valueAsNumber")
    )
    return Div(price), Div(quantity), Div(o)

In [6]:
srv = JupyUvi(app)

### Conditional class

In [29]:
@rt('/')
def get():
    i = Label("Age: ", Input(id="age", type="number", value="0"))
    o = P(
        "Adult content",
        hx_live="this.classList.toggle('warn', q('#age').valueAsNumber < 18)"
    )
    return Style(".warn { color: red; font-weight: bold; }"), Div(i), Div(o)

### Live Filter

In [5]:
@rt('/')
def get():
    i = Input(id="filter", placeholder="filter")
    ls = Ul(Li("apple"), Li("apricot", Li("banana")))
    o = Div(
        hx_live="""
            let f = q('#filter').value.toLowerCase();
            for (let li of q('li')) li.hidden = !li.textContent.toLowerCase().includes(f);
        """
    )
    return (Div(i), Div(ls), o)

## Tab selection

In [14]:
@rt('/')
def get():
    tab_script = "take('selected', q('closest nav').q('button'))"
    return Div(
        Style("""
            nav { display: flex; gap: .5rem; }
            nav button { background: white; color: #333; }
            nav button.selected { background: #111; color: white; }
        """),
        Nav(
            Button('A', cls='selected', hx_on_click=tab_script),
            Button('B', hx_on_click=tab_script),
            Button('C', hx_on_click=tab_script),
        ),
        hx_ext='hx-live'
    )


In [10]:
srv = JupyUvi(app)


### Debounced live search


In [16]:
@rt('/')
def get():
    return Div(
        Input(id='q', placeholder='search'),
        Output(
            hx_live="""
                let term = q('#q').value;
                if (!term) { this.textContent = ''; return; }
                await debounce(250);
                this.textContent = await fetch('/search?q=' + encodeURIComponent(term))
                                          .then(r => r.text());
            """
        ),
        hx_ext='hx-live'
    )

@rt('/search')
def get(q: str = ''):
    items = ['apple', 'apricot', 'banana', 'blueberry', 'cherry']
    matches = [item for item in items if q.lower() in item]
    return ', '.join(matches) or 'No matches'


In [ ]:
srv = JupyUvi(app)
